In [1]:
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_curve, roc_auc_score, average_precision_score

DATA = "s3://sagemaker-salesmesh-myra/processed/"
train = pd.read_csv(DATA + "train.csv")
test = pd.read_csv(DATA + "test.csv")

LABEL = "fraud_bool"
CAT = ["source", "device_os"]
CORE = ["name_email_similarity", "customer_age", "prev_address_months_count",
        "current_address_months_count", "date_of_birth_distinct_emails_4w",
        "email_is_free", "phone_home_valid", "phone_mobile_valid", "foreign_request",
        "source", "session_length_in_minutes", "device_os", "keep_alive_session",
        "device_distinct_emails_8w"]
SCALE = ["velocity_6h", "velocity_24h", "velocity_4w", "zip_count_4w"]
FULL = CORE + SCALE

y_train, y_test = train[LABEL], test[LABEL]
print(train.shape, test.shape)

(794989, 19) (205011, 19)


In [3]:
def recall_at_fpr(y_true, y_prob, max_fpr=0.05):
    """Recall (TPR) at the highest threshold where FPR <= max_fpr. Also returns that threshold."""
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    i = np.searchsorted(fpr, max_fpr, side="right") - 1
    return tpr[i], thr[i]

def build_pipeline(model, features, scale):
    num = [c for c in features if c not in CAT]
    cat = [c for c in features if c in CAT]
    num_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    prep = ColumnTransformer([
        ("num", Pipeline(num_steps), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])
    return Pipeline([("prep", prep), ("model", model)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [4]:
lr_pipe = build_pipeline(
    LogisticRegression(class_weight="balanced", max_iter=2000),
    CORE, scale=True)

lr_search = GridSearchCV(
    lr_pipe, {"model__C": [0.01, 0.1, 1, 10]},
    scoring="average_precision", cv=cv, n_jobs=-1, verbose=1)

t = time.time()
lr_search.fit(train[CORE], y_train)
print(f"Time: {(time.time() - t) / 60:.1f} min")
print("Best:", lr_search.best_params_, "CV PR-AUC:", round(lr_search.best_score_, 4))

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Time: 0.5 min
Best: {'model__C': 0.01} CV PR-AUC: 0.0701


In [5]:
rf_pipe = build_pipeline(
    RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample",
                           max_samples=0.3, n_jobs=-1, random_state=42),
    CORE, scale=False)

rf_search = GridSearchCV(
    rf_pipe,
    {"model__max_depth": [8, 12, 16], "model__min_samples_leaf": [50, 200]},
    scoring="average_precision", cv=cv, n_jobs=1, verbose=2)

t = time.time()
rf_search.fit(train[CORE], y_train)
print(f"Time: {(time.time() - t) / 60:.1f} min")
print("Best:", rf_search.best_params_, "CV PR-AUC:", round(rf_search.best_score_, 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
[CV] END .....model__max_depth=8, model__min_samples_leaf=50; total time=  23.2s
[CV] END .....model__max_depth=8, model__min_samples_leaf=50; total time=  23.1s
[CV] END .....model__max_depth=8, model__min_samples_leaf=50; total time=  23.9s
[CV] END .....model__max_depth=8, model__min_samples_leaf=50; total time=  23.9s
[CV] END .....model__max_depth=8, model__min_samples_leaf=50; total time=  23.7s
[CV] END ....model__max_depth=8, model__min_samples_leaf=200; total time=  22.9s
[CV] END ....model__max_depth=8, model__min_samples_leaf=200; total time=  23.8s
[CV] END ....model__max_depth=8, model__min_samples_leaf=200; total time=  23.0s
[CV] END ....model__max_depth=8, model__min_samples_leaf=200; total time=  23.3s
[CV] END ....model__max_depth=8, model__min_samples_leaf=200; total time=  23.0s
[CV] END ....model__max_depth=12, model__min_samples_leaf=50; total time=  25.9s
[CV] END ....model__max_depth=12, model__min_samp

In [6]:
lr_full = build_pipeline(
    LogisticRegression(class_weight="balanced", max_iter=2000,
                       C=lr_search.best_params_["model__C"]),
    FULL, scale=True).fit(train[FULL], y_train)

rf_full = build_pipeline(
    RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample",
                           max_samples=0.3, n_jobs=-1, random_state=42,
                           max_depth=rf_search.best_params_["model__max_depth"],
                           min_samples_leaf=rf_search.best_params_["model__min_samples_leaf"]),
    FULL, scale=False).fit(train[FULL], y_train)

In [7]:
models = {
    "LR core":  (lr_search.best_estimator_, CORE),
    "RF core":  (rf_search.best_estimator_, CORE),
    "LR full":  (lr_full, FULL),
    "RF full":  (rf_full, FULL),
}

rows = []
for name, (m, feats) in models.items():
    p = m.predict_proba(test[feats])[:, 1]
    rec, thr = recall_at_fpr(y_test, p)
    rows.append({"model": name,
                 "recall@5%FPR": round(rec, 4),
                 "threshold": round(thr, 4),
                 "ROC-AUC": round(roc_auc_score(y_test, p), 4),
                 "PR-AUC": round(average_precision_score(y_test, p), 4)})

results = pd.DataFrame(rows).set_index("model")
results

,recall@5%FPR,threshold,ROC-AUC,PR-AUC
model,,,,
LR core,0.3641,0.7705,0.8250,0.0925
RF core,0.3589,0.6655,0.8266,0.0912
LR full,0.3621,0.7578,0.8246,0.0926
RF full,0.3628,0.5612,0.8289,0.0883


In [8]:
import os, s3fs
os.makedirs("../artifacts", exist_ok=True)
fs = s3fs.S3FileSystem()

for name, (m, feats) in models.items():
    fname = name.replace(" ", "_").lower() + ".joblib"
    joblib.dump({"model": m, "features": feats}, f"../artifacts/{fname}")
    fs.put(f"../artifacts/{fname}", f"sagemaker-salesmesh-myra/models/candidates/{fname}")

results.to_csv("../artifacts/results_test.csv")
print("Saved.")

Saved.


In [9]:
lr_search2 = GridSearchCV(
    build_pipeline(LogisticRegression(class_weight="balanced", max_iter=2000), CORE, scale=True),
    {"model__C": [0.0001, 0.001, 0.01]},
    scoring="average_precision", cv=cv, n_jobs=-1, verbose=1)
lr_search2.fit(train[CORE], y_train)
print("Old:", lr_search.best_params_, round(lr_search.best_score_, 4))
print("New:", lr_search2.best_params_, round(lr_search2.best_score_, 4))

Fitting 5 folds for each of 3 candidates, totalling 15 fits
Old: {'model__C': 0.01} 0.0701
New: {'model__C': 0.001} 0.0702


In [10]:
rf_search2 = GridSearchCV(
    build_pipeline(RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample",
                                          max_samples=0.3, n_jobs=-1, random_state=42),
                   CORE, scale=False),
    {"model__max_depth": [16, 24, None], "model__min_samples_leaf": [20, 50]},
    scoring="average_precision", cv=cv, n_jobs=1, verbose=2)
rf_search2.fit(train[CORE], y_train)
print("Old:", rf_search.best_params_, round(rf_search.best_score_, 4))
print("New:", rf_search2.best_params_, round(rf_search2.best_score_, 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
[CV] END ....model__max_depth=16, model__min_samples_leaf=20; total time=  26.1s
[CV] END ....model__max_depth=16, model__min_samples_leaf=20; total time=  25.7s
[CV] END ....model__max_depth=16, model__min_samples_leaf=20; total time=  25.7s
[CV] END ....model__max_depth=16, model__min_samples_leaf=20; total time=  25.5s
[CV] END ....model__max_depth=16, model__min_samples_leaf=20; total time=  25.8s
[CV] END ....model__max_depth=16, model__min_samples_leaf=50; total time=  24.8s
[CV] END ....model__max_depth=16, model__min_samples_leaf=50; total time=  25.4s
[CV] END ....model__max_depth=16, model__min_samples_leaf=50; total time=  24.7s
[CV] END ....model__max_depth=16, model__min_samples_leaf=50; total time=  25.1s
[CV] END ....model__max_depth=16, model__min_samples_leaf=50; total time=  25.3s
[CV] END ....model__max_depth=24, model__min_samples_leaf=20; total time=  26.0s
[CV] END ....model__max_depth=24, model__min_samp